# HW05: Workspace Dependence & the Via-Point Timing Problem

**Computational Sensorimotor Control — Week 5**

In Lab 5 you discovered that EPH produces curved, workspace-dependent hand paths while minimum jerk and VITE produce straight ones (§3, §6). You also saw that EPH undershoots via-points and requires an explicit dwell whose duration the programmer must choose (§8). This homework quantifies both problems systematically.

**Three experiments:**
1. **Path curvature** — measure how much EPH paths deviate from straight across arm configurations
2. **EPH dwell sweep** — discover the tradeoff between via-point accuracy and movement speed
3. **VITE robustness** — show that VITE handles via-points regardless of switch timing

In [ ]:
# Install the smc library (only needs to run once per session)
!pip3 install git+https://github.com/tarkeshsingh/computational-sensorimotor-control.git#subdirectory=smc -q

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

from smc import (
    Arm, make_muscles, simulate_lambda, Q_REF, L1, L2,
    rk4_step, MU_LAMBDA,
)

arm = Arm()
ik = arm.inverse_kinematics
start_hand = arm.forward_kinematics(Q_REF)

# -- Lab 04/05 helpers --
def lambda_for_posture(q_target, C=0.25):
    muscles = make_muscles()
    return np.array([m.length(q_target) - (abs(m.r_sh) + abs(m.r_el)) * C
                     for m in muscles])

def make_ramp(lam_init, lam_final, t_start=0.05, duration=0.30):
    def lam_fn(t):
        if t < t_start:
            return lam_init.copy()
        elif t < t_start + duration:
            return lam_init + (t - t_start) / duration * (lam_final - lam_init)
        else:
            return lam_final.copy()
    return lam_fn

def minimum_jerk(p0, pf, T, dt=0.001):
    t = np.arange(0, T + dt, dt); tau = t / T
    s  = 10*tau**3 - 15*tau**4 + 6*tau**5
    sd = (30*tau**2 - 60*tau**3 + 30*tau**4) / T
    d = (pf - p0)[None, :]
    return t, p0[None, :] + s[:, None] * d, sd[:, None] * d

DT_VITE = 0.0002
def vite_deriv(state, target, G, alpha):
    Px, Py, Vx, Vy = state
    return np.array([Vx, Vy,
                     alpha * (-Vx + G * (target[0] - Px)),
                     alpha * (-Vy + G * (target[1] - Py))])

def sim_vite(p0, target, G_amp, T=1.2, dt=DT_VITE):
    alpha = 4.0 * G_amp
    state = np.array([p0[0], p0[1], 0.0, 0.0])
    n = int(T / dt)
    ts = np.zeros(n); pos = np.zeros((n, 2)); vel = np.zeros((n, 2))
    for i in range(n):
        ts[i] = i * dt; pos[i] = state[:2]; vel[i] = state[2:]
        state = rk4_step(state,
            lambda s, _G=G_amp, _a=alpha: vite_deriv(s, target, _G, _a), dt)
    return ts, pos, vel

def max_perpendicular_deviation(path, start, end):
    """Max perpendicular distance from a hand path to the start-to-end line (cm)."""
    d = end - start
    d_norm = d / np.linalg.norm(d)
    deviations = []
    for p in path:
        v = p - start
        proj = np.dot(v, d_norm) * d_norm
        perp = np.linalg.norm(v - proj)
        deviations.append(perp)
    return np.max(deviations) * 100

# -- Starting configurations --
CONFIGS = {
    'Q_REF (55, 75)':     Q_REF.copy(),
    'Extended (35, 45)':   np.radians([35, 45]),
    'Flexed (75, 110)':    np.radians([75, 110]),
    'Abducted (40, 80)':   np.radians([40, 80]),
}

REACH_RADIUS = 0.08
C_CMD = 0.25
COLORS_8 = ['#E74C3C','#E67E22','#27AE60','#2E86AB','#3498db','#8e44ad','#E8735A','#7f8c8d']

print("Starting configurations:")
for name, q in CONFIGS.items():
    h = arm.forward_kinematics(q)
    J = arm.jacobian(q)
    cond = np.linalg.cond(J)
    print(f"  {name}: hand at ({h[0]*100:.1f}, {h[1]*100:.1f}) cm, J cond = {cond:.2f}")

---
## Part 1: Path Curvature Across the Workspace (40 pts)

### Task 1.1 — Curvature Measurement (30 pts)

For each of 4 starting configurations, run EPH, min-jerk, and VITE to 8 center-out targets (8 cm).
Compute `max_perpendicular_deviation` for each reach.

**Note:** From the Extended (35°, 45°) configuration, the 45° and 90° targets are **outside the reachable workspace**. The code handles this with `try/except` and records `np.nan` for those reaches. This is expected — the arm is nearly fully extended and cannot reach farther in those directions.

**Your tasks (fill in the 5 marked lines):**
1. Compute the EPH target λ values (`lf = ...`)
2. Run the EPH simulation (`simulate_lambda(...)`)
3. Run the min-jerk trajectory (`minimum_jerk(...)`)
4. Run the VITE simulation (`sim_vite(...)`)
5. Compute curvature for each controller (`max_perpendicular_deviation(...)`)

### Question 1.1 (10 pts)

**(a)** Which controller shows the most curvature? Does the amount depend on starting configuration? **(b)** Does higher Jacobian condition number predict more EPH curvature? **(c)** In 2–3 sentences: if EPH’s curved paths are caused by the nonlinear Jacobian mapping (§3), why don’t VITE paths curve too?

In [ ]:
angles_8 = np.arange(0, 360, 45)
results = {}

fig_paths, axes_p = plt.subplots(4, 3, figsize=(15, 18))
controllers = ['EPH', 'Min-Jerk', 'VITE']

for ci, (cfg_name, q_start) in enumerate(CONFIGS.items()):
    h_start = arm.forward_kinematics(q_start)
    curvatures = {'EPH': [], 'Min-Jerk': [], 'VITE': []}

    for di, deg in enumerate(angles_8):
        tgt = h_start + REACH_RADIUS * np.array([np.cos(np.radians(deg)),
                                                   np.sin(np.radians(deg))])
        # Check reachability
        try:
            tq = ik(tgt[0], tgt[1])
        except:
            for ctrl in curvatures: curvatures[ctrl].append(np.nan)
            continue

        # ── EPH ──
        try:
            li = lambda_for_posture(q_start, C_CMD)
            lf = None  # TODO: compute lambda values for target posture tq
            te, se, he, _ = None, None, None, None  # TODO: run simulate_lambda with make_ramp(li, lf, duration=0.15), T=1.0, q0=q_start
            curvatures['EPH'].append(max_perpendicular_deviation(he, h_start, tgt))
            axes_p[ci, 0].plot(he[:, 0]*100, he[:, 1]*100, color=COLORS_8[di], lw=1.2)
        except:
            curvatures['EPH'].append(np.nan)

        # ── Min-Jerk ──
        tm, pm, vm = None, None, None  # TODO: call minimum_jerk(h_start, tgt, 0.5)
        curvatures['Min-Jerk'].append(max_perpendicular_deviation(pm, h_start, tgt))
        axes_p[ci, 1].plot(pm[:, 0]*100, pm[:, 1]*100, color=COLORS_8[di], lw=1.2)

        # ── VITE ──
        tv, pv, vv = None, None, None  # TODO: call sim_vite(h_start, tgt, G_amp=10, T=1.0)
        curvatures['VITE'].append(max_perpendicular_deviation(pv, h_start, tgt))
        axes_p[ci, 2].plot(pv[:, 0]*100, pv[:, 1]*100, color=COLORS_8[di], lw=1.2)

        for j in range(3):
            axes_p[ci, j].plot(tgt[0]*100, tgt[1]*100, 'o', color=COLORS_8[di], ms=3)

    for j, ctrl in enumerate(controllers):
        axes_p[ci, j].plot(h_start[0]*100, h_start[1]*100, 'ko', ms=5, zorder=5)
        axes_p[ci, j].set_aspect('equal')
        if ci == 0: axes_p[ci, j].set_title(ctrl, fontweight='bold')
        if j == 0: axes_p[ci, j].set_ylabel(cfg_name, fontsize=9)

    results[cfg_name] = curvatures
    J = arm.jacobian(q_start)
    eph_mean = np.nanmean(curvatures['EPH'])
    print(f"{cfg_name}: cond(J) = {np.linalg.cond(J):.2f}, EPH mean curv = {eph_mean:.2f} cm")

fig_paths.suptitle('Figure 1: Hand Paths (4 Configs x 3 Controllers)', fontweight='bold', y=1.01)
plt.tight_layout()

# ── Summary bar chart (EPH only — MJ and VITE curvature is near-zero) ──
fig_bar, (ax_eph, ax_task) = plt.subplots(1, 2, figsize=(13, 5))

# Left: EPH curvature by config and direction
x = np.arange(len(CONFIGS))
for ci, cfg in enumerate(CONFIGS):
    vals = [v for v in results[cfg]['EPH'] if not np.isnan(v)]
    ax_eph.bar(ci, np.mean(vals), yerr=np.std(vals), color='#E74C3C',
               alpha=0.8, capsize=5, width=0.6)
ax_eph.set_xticks(x); ax_eph.set_xticklabels(CONFIGS.keys(), fontsize=8, rotation=15)
ax_eph.set_ylabel('Max Curvature (cm)'); ax_eph.set_title('A. EPH Curvature by Configuration', fontweight='bold')

# Right: All 3 controllers (note scale difference)
width = 0.25
colors_ctrl = ['#E74C3C', '#F39C12', '#2E86AB']
for ki, ctrl in enumerate(controllers):
    means = [np.nanmean(results[cfg][ctrl]) for cfg in CONFIGS]
    stds  = [np.nanstd(results[cfg][ctrl]) for cfg in CONFIGS]
    ax_task.bar(x + ki*width, means, width, yerr=stds, label=ctrl,
                color=colors_ctrl[ki], alpha=0.8, capsize=4)
ax_task.set_xticks(x + width); ax_task.set_xticklabels(CONFIGS.keys(), fontsize=8, rotation=15)
ax_task.set_ylabel('Max Curvature (cm)'); ax_task.legend()
ax_task.set_title('B. All Controllers (note: MJ and VITE near zero)', fontweight='bold')

fig_bar.suptitle('Figure 2: Path Curvature Summary', fontweight='bold', y=1.02)
plt.tight_layout()

### Question 1.1

*Write your answer here.*

---
## Part 2: The Via-Point Timing Problem (40 pts)

### Task 2.1 — EPH Dwell Duration Sweep (20 pts)

Via-point reach from Lab 05 Part 8: start at Q_REF, via at start + (4, 6) cm, final at start + (8, 0) cm.
EPH uses 150 ms ramps. Vary the dwell duration from 0 to 300 ms.

### Task 2.2 — VITE Switch-Time Robustness (15 pts)

Same via-point reach with VITE (G = 10). Vary switch time from 50 to 400 ms.

**Your tasks (fill in the 4 marked lines):**
1. Build the EPH ramp function for the via-point → final segment
2. Compute closest approach to the via-point for EPH
3. Set the VITE target based on whether t is before or after the switch time
4. Compute closest approach to the via-point for VITE

### Question 2.1 (5 pts)

**(a)** How sensitive is EPH’s via-point accuracy to dwell duration? Is there a sweet spot? **(b)** How sensitive is VITE to switch time? **(c)** In 2–3 sentences: both EPH and VITE need a timing parameter. What is the *architectural* difference — why is VITE more robust?

In [ ]:
via_pt   = start_hand + np.array([0.04, 0.06])
final_pt = start_hand + np.array([0.08, 0.0])

via_q   = ik(via_pt[0], via_pt[1])
final_q = ik(final_pt[0], final_pt[1])
li = lambda_for_posture(Q_REF)
lv = lambda_for_posture(via_q)
lf = lambda_for_posture(final_q)

# ════════════════════════════════════════════════════════════════════════
# TASK 2.1: EPH dwell sweep
# ════════════════════════════════════════════════════════════════════════
dwell_durations = [0, 0.025, 0.050, 0.075, 0.100, 0.150, 0.200, 0.300]
eph_closest = []; eph_mvt_time = []; eph_paths = []
cmap_dwell = plt.cm.plasma(np.linspace(0.1, 0.9, len(dwell_durations)))

for dwell in dwell_durations:
    r1_start, r1_end = 0.05, 0.20          # Ramp 1: 150 ms
    dwell_end = r1_end + dwell              # Dwell period
    r2_end = dwell_end + 0.15              # Ramp 2: 150 ms

    def mk_ramp(r1s, r1e, de, r2e):
        def fn(t):
            if t < r1s:    return li.copy()
            elif t < r1e:  return li + (t - r1s) / 0.15 * (lv - li)
            elif t < de:   return lv.copy()                          # dwell at via
            elif t < r2e:  return None  # TODO: interpolate from lv to lf over 150 ms
            else:          return lf.copy()
        return fn

    lam_fn = mk_ramp(r1_start, r1_end, dwell_end, r2_end)
    te, se, he, _ = simulate_lambda(lam_fn, T=max(1.5, r2_end + 0.8))
    spe = np.zeros(len(te))
    spe[1:] = np.linalg.norm(np.diff(he, axis=0) / 0.0001, axis=1)

    # Closest approach to via-point
    dists = None  # TODO: compute distance from each hand position to via_pt, in cm
    eph_closest.append(dists.min())

    # Movement time: onset (>1 cm/s) to settle (<1 cm/s)
    moving = spe * 100 > 1.0
    if moving.any():
        onset = te[np.argmax(moving)]
        last_fast = np.where(moving)[0][-1]
        eph_mvt_time.append((te[last_fast] - onset) * 1000)
    else:
        eph_mvt_time.append(0)
    eph_paths.append(he)

# ── Figure 3: EPH hand paths ──
fig3, ax3 = plt.subplots(figsize=(7, 6))
for di, (dwell, he) in enumerate(zip(dwell_durations, eph_paths)):
    ax3.plot(he[:, 0]*100, he[:, 1]*100, color=cmap_dwell[di], lw=1.5,
             label=f'{dwell*1000:.0f} ms')
ax3.plot(*start_hand*100, 'ko', ms=8, zorder=5)
ax3.plot(via_pt[0]*100, via_pt[1]*100, 's', color='#666', ms=10, label='Via', zorder=5)
ax3.plot(final_pt[0]*100, final_pt[1]*100, 'r*', ms=12, label='Target', zorder=5)
ax3.set_aspect('equal'); ax3.legend(fontsize=8, title='Dwell', title_fontsize=9)
ax3.set_xlabel('x (cm)'); ax3.set_ylabel('y (cm)')
ax3.set_title('Figure 3: EPH Via-Point Paths vs. Dwell Duration', fontweight='bold')
plt.tight_layout()

# ════════════════════════════════════════════════════════════════════════
# TASK 2.2: VITE switch-time sweep
# ════════════════════════════════════════════════════════════════════════
switch_times = np.arange(0.05, 0.45, 0.05)
vite_closest = []; vite_mvt_time = []

for sw_t in switch_times:
    alpha = 40.0; G = 10
    state = np.array([start_hand[0], start_hand[1], 0.0, 0.0])
    n = int(1.2 / DT_VITE)
    pos_v = np.zeros((n, 2)); vel_v = np.zeros((n, 2)); ts_v = np.zeros(n)
    for i in range(n):
        t = i * DT_VITE; ts_v[i] = t
        pos_v[i] = state[:2]; vel_v[i] = state[2:]
        tg = None  # TODO: set target to via_pt before sw_t, final_pt after
        state = rk4_step(state,
            lambda s, _tg=tg: vite_deriv(s, _tg, G, alpha), DT_VITE)

    spdv = np.linalg.norm(vel_v, axis=1) * 100
    dists_v = None  # TODO: compute distance from each position to via_pt, in cm
    vite_closest.append(dists_v.min())

    moving_v = spdv > 1.0
    if moving_v.any():
        onset_v = ts_v[np.argmax(moving_v)]
        last_v = np.where(moving_v)[0][-1]
        vite_mvt_time.append((ts_v[last_v] - onset_v) * 1000)
    else:
        vite_mvt_time.append(0)

# ── Figure 4: Summary comparison ──
fig4, (ax_ca, ax_mt) = plt.subplots(1, 2, figsize=(13, 5))

ax_ca.plot([d*1000 for d in dwell_durations], eph_closest, 'o-',
           color='#E74C3C', lw=2, ms=7, label='EPH (dwell)')
ax_ca.plot(switch_times*1000, vite_closest, 's--',
           color='#2E86AB', lw=2, ms=7, label='VITE (switch)')
ax_ca.set_xlabel('Dwell / Switch Time (ms)'); ax_ca.set_ylabel('Closest Approach (cm)')
ax_ca.legend(); ax_ca.set_title('A. Via-Point Accuracy', fontweight='bold')

ax_mt.plot([d*1000 for d in dwell_durations], eph_mvt_time, 'o-',
           color='#E74C3C', lw=2, ms=7, label='EPH (dwell)')
ax_mt.plot(switch_times*1000, vite_mvt_time, 's--',
           color='#2E86AB', lw=2, ms=7, label='VITE (switch)')
ax_mt.set_xlabel('Dwell / Switch Time (ms)'); ax_mt.set_ylabel('Movement Time (ms)')
ax_mt.legend(); ax_mt.set_title('B. Total Movement Time', fontweight='bold')

fig4.suptitle('Figure 4: EPH Dwell vs. VITE Switch', fontweight='bold', y=1.02)
plt.tight_layout()

print(f"EPH closest range:  {min(eph_closest):.2f} - {max(eph_closest):.2f} cm")
print(f"VITE closest range: {min(vite_closest):.2f} - {max(vite_closest):.2f} cm")

### Question 2.1

*Write your answer here.*

---
## Part 3: Write-Up (20 pts)

Write a short Methods & Results section (300–500 words) as if for a journal paper. Include:

- **Methods:** Briefly describe the three controllers (cite Flash & Hogan 1985, Bullock & Grossberg 1988, Gribble et al. 1998), the curvature experiment (4 configurations × 8 directions × 3 controllers), and the via-point timing experiment (8 EPH dwell durations, 8 VITE switch times).

- **Results:** Summarize your key findings from Parts 1–2. Reference your figures by number. State which controller showed the most curvature, whether it correlated with Jacobian condition number, and how sensitive each controller was to via-point timing.

- **Connection to Lecture §3:** In 2–3 sentences, relate your curvature findings to the Discussion in §3 (“Are EPH’s Curved Paths Really Wrong?”). Do your results support the idea that EPH’s curvatures match the systematic direction-dependent curvatures observed experimentally (Atkeson & Hollerbach 1985, Flash 1987)?

*Write your answer here.*